# CrossMed 01 - Prepare datasets
Create leakage-safe BUS-BRA splits and locked BrEaST/BUSI external-test manifests.

In [ ]:
from pathlib import Path
import pandas as pd
from crossmed.prepare import prepare_all

## Set dataset paths
Change these paths to match your extracted downloads.

In [ ]:
DATA_ROOT = Path('../data').resolve()
RAW = DATA_ROOT / 'raw'
BUS_BRA_ROOT = RAW / 'busbra'
BUS_BRA_METADATA = BUS_BRA_ROOT / 'bus_data.csv'
BREAST_ROOT = RAW / 'breast'
BREAST_XLSX = BREAST_ROOT / 'BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx'
BUSI_ROOT = RAW / 'busi'
MANIFEST_DIR = DATA_ROOT / 'manifests'

In [ ]:
summary = prepare_all(
    BUS_BRA_ROOT, BUS_BRA_METADATA, BREAST_ROOT, BREAST_XLSX, BUSI_ROOT,
    output_dir=MANIFEST_DIR, seed=566, data_root=DATA_ROOT
)
pd.DataFrame(summary).T

## Audit manifests
Confirm class counts, readable paths, no masks, and no patient overlap before training.

In [ ]:
frames = {p.stem: pd.read_csv(p) for p in MANIFEST_DIR.glob('*.csv')}
for name, frame in frames.items():
    missing = (~frame.image_path.map(lambda x: (DATA_ROOT / x).is_file())).sum()
    masks = frame.image_path.str.contains('_mask', case=False).sum()
    print(name, frame.shape, frame.label.value_counts().to_dict(), 'missing=', missing, 'masks=', masks)

In [ ]:
train_patients = set(frames['busbra_train'].patient_id.astype(str))
val_patients = set(frames['busbra_val'].patient_id.astype(str))
test_patients = set(frames['busbra_test'].patient_id.astype(str))
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)
print('Patient split audit passed')